# 🏡 Min-Max Normalization Workshop
## Team Name/Number: Group 1
## Team Members: Sultan Akinola Atanda · John Buni · Eche Oji · Antonio Sainz
---

## ❗ Why We Normalize: The Problem with Raw Feature Scales

In housing data, features like `Price` and `Lot_Size` can have values in the hundreds of thousands, while others like `Num_Bedrooms` range from 1 to 5. This creates problems when we use algorithms that depend on numeric magnitudes.

---

### ⚠️ What Goes Wrong Without Normalization

---

### 1. 🧭 K-Nearest Neighbors (KNN)

KNN uses the **Euclidean distance** formula:

$$
d = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2 + \cdots}
$$

**Example:**

- $ \text{Price}_1 = 650{,}000, \quad \text{Price}_2 = 250{,}000 $
- $ \text{Bedrooms}_1 = 3, \quad \text{Bedrooms}_2 = 2 $

Now compute squared differences:

$$
(\text{Price}_1 - \text{Price}_2)^2 = (650{,}000 - 250{,}000)^2 = (400{,}000)^2 = 1.6 \times 10^{11}
$$
$$
(\text{Bedrooms}_1 - \text{Bedrooms}_2)^2 = (3 - 2)^2 = 1
$$

➡️ **Price dominates the distance calculation**, making smaller features like `Bedrooms` irrelevant.

---

### 2. 📉 Linear Regression

Linear regression estimates:

$$
y = \beta_1 \cdot \text{Price} + \beta_2 \cdot \text{Bedrooms} + \beta_3 \cdot \text{Lot\_Size} + \epsilon
$$

If `Price` has very large values:
- Gradient updates for $ \beta_1 $ will be **much larger**
- Gradient updates for $ \beta_2 $ (Bedrooms) will be **very small**

➡️ The model overfits high-magnitude features like `Price`.

---

### 3. 🧠 Neural Networks

A single neuron computes:

$$
z = w_1 \cdot \text{Price} + w_2 \cdot \text{Bedrooms} + w_3 \cdot \text{Lot\_Size}
$$

If:

- $ \text{Price} = 650{,}000 $
- $ \text{Bedrooms} = 3 $
- $ \text{Lot\_Size} = 8{,}000 $

Then:

$$
z \approx w_1 \cdot 650{,}000 + w_2 \cdot 3 + w_3 \cdot 8{,}000
$$

➡️ Even with equal weights, `Price` contributes **most of the activation**, making it difficult for the network to learn from other features.

---

### ✅ Solution: Min-Max Normalization

We apply the transformation:

$$
x_{\text{normalized}} = \frac{x - x_{\text{min}}}{x_{\text{max}} - x_{\text{min}}}
$$

This scales all features to a common range (typically $[0, 1]$).

| Feature      | Raw Value | Min     | Max     | Normalized Value |
|--------------|-----------|---------|---------|------------------|
| Price        | 650,000   | 250,000 | 800,000 | 0.72             |
| Bedrooms     | 3         | 1       | 5       | 0.50             |
| Lot_Size     | 8,000     | 3,000   | 10,000  | 0.714            |

➡️ Now, **each feature contributes fairly** to model training or distance comparisons.

---

## 📌 Use Case: Housing Data
We are normalizing features from a real estate dataset to prepare it for machine learning analysis.

In [22]:
import pandas as pd
df = pd.read_csv('./data/housing_data.csv')
df.head()

,House_ID,Price,Area_sqft,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size
0,H100000,574507,1462,3,3,2002,4878
1,H100001,479260,1727,2,2,1979,4943
2,H100002,597153,1403,5,2,1952,5595
3,H100003,728454,1646,5,2,1992,9305
4,H100004,464876,853,1,1,1956,7407


In [23]:
from src.normalizer import MinMaxNormalizer
from src.pca_analyzer import PCAAnalyzer
from src.feature_insights import FeatureInsights
from src.visualizer import HousingVisualizer

NUMERIC = ['Price', 'Area_sqft', 'Num_Bedrooms', 'Num_Bathrooms', 'Year_Built', 'Lot_Size']
viz = HousingVisualizer()

print(f"{df.shape[0]:,} houses · {df.shape[1]} columns · missing values: {df.isna().sum().sum()} · duplicate rows: {df.duplicated().sum()}")
df[NUMERIC].describe().T.round(1)

2,000 houses · 7 columns · missing values: 0 · duplicate rows: 0


,count,mean,std,min,25%,50%,75%,max
Price,2000.0,506896.1,147878.6,100000.0,406600.2,506703.0,602445.8,1077909.0
Area_sqft,2000.0,1796.5,502.2,400.0,1445.0,1799.5,2132.0,3763.0
Num_Bedrooms,2000.0,3.0,1.4,1.0,2.0,3.0,4.0,5.0
Num_Bathrooms,2000.0,2.0,0.8,1.0,1.0,2.0,3.0,3.0
Year_Built,2000.0,1985.7,21.2,1950.0,1967.0,1986.0,2003.0,2022.0
Lot_Size,2000.0,6025.2,2008.5,1000.0,4664.0,6010.5,7414.0,13088.0


### 🔎 Step 1 — Implement Min-Max Normalization on the Housing Dataset

In [24]:
normalizer = MinMaxNormalizer(NUMERIC)
df_scaled = normalizer.fit_transform(df)

df_norm = df_scaled[[f'{c}_norm' for c in NUMERIC]].copy()
df_norm.columns = NUMERIC

df_scaled[['House_ID'] + [f'{c}_norm' for c in NUMERIC]].head()

,House_ID,Price_norm,Area_sqft_norm,Num_Bedrooms_norm,Num_Bathrooms_norm,Year_Built_norm,Lot_Size_norm
0,H100000,0.485226,0.315789,0.50,1.0,0.722222,0.320814
1,H100001,0.387827,0.394588,0.25,0.5,0.402778,0.326191
2,H100002,0.508384,0.298246,1.00,0.5,0.027778,0.380129
3,H100003,0.642651,0.370503,1.00,0.5,0.583333,0.687045
4,H100004,0.373119,0.134701,0.00,0.0,0.083333,0.530030


`Year_Built` is not in the original list, but we normalize it too because we decided to keep it as a PCA feature (see the PCA section).
Below is the same calculation as the theory table, done for the first house in the file. Every value can be checked by hand with $(x - x_{min}) / (x_{max} - x_{min})$.

In [25]:
normalizer.worked_example(df.iloc[0])

,Raw,Min,Max,Normalized
Feature,,,,
Price,574507,100000,1077909,0.4852
Area_sqft,1462,400,3763,0.3158
Num_Bedrooms,3,1,5,0.5000
Num_Bathrooms,3,1,3,1.0000
Year_Built,2002,1950,2022,0.7222
Lot_Size,4878,1000,13088,0.3208


In [26]:
from sklearn.preprocessing import MinMaxScaler

check = MinMaxScaler().fit_transform(df[NUMERIC])
gap = abs(check - df_norm.values).max()
print(f"Largest difference vs sklearn MinMaxScaler: {gap:.2e}")
normalizer.verify(df_scaled)

Largest difference vs sklearn MinMaxScaler: 2.00e-15


,min,max
Price,0.0,1.0
Area_sqft,0.0,1.0
Num_Bedrooms,0.0,1.0
Num_Bathrooms,0.0,1.0
Year_Built,0.0,1.0
Lot_Size,0.0,1.0


**Before vs after normalization.** The charts below show what changed.

In [27]:
ranges_norm = normalizer.summary().assign(Range=1.0)
fig = viz.scale_gap(normalizer.summary(), ranges_norm)
fig.show()

In [28]:
fig = viz.distributions(df[NUMERIC], df_norm)
fig.show()

In [29]:
shares = FeatureInsights(df[NUMERIC], df_norm).distance_share()
fig = viz.distance_share(shares)
fig.show()
(shares * 100).round(2).rename(columns=lambda c: f'{c} (%)')

,Raw (%),Min-max (%)
Price,99.98,5.25
Area_sqft,0.00,4.95
Num_Bedrooms,0.00,27.90
Num_Bathrooms,0.00,36.08
Year_Built,0.00,19.85
Lot_Size,0.02,5.98


### 🔎 Talking Points #1 — Why normalization matters for KNN, gradient-based learning and housing features with different scales

- **KNN:** Without scaling, Price makes up **99.98%** of the distance between two houses. KNN just finds houses with a similar price and ignores bedrooms and bathrooms. After scaling, every feature counts.
- **Gradient descent:** Big inputs like Price (~500,000) get much bigger updates than small ones like bathrooms (1–3). Training becomes slow and unstable. With all features between 0 and 1, it learns evenly.
- **Housing features:** Price spans ~978,000 while bathrooms span only 2. Min-max puts them all on the same 0–1 scale. One catch: a single very expensive house sets the top of the scale, so outliers matter.


## 🧩 Challenge Extension: After Normalization, Which Features Matter Most?

You’ve normalized the housing features so they share a common scale.  
Now comes a common next step in ML workflows:

> **How do we identify the most important directions (principal components) in the data—and how might these relate to a target variable like `Price`?**

This introduces **Principal Component Analysis (PCA)**.

---

## 📚 PCA Theory (Conceptual)

### What PCA *is*
PCA is an **unsupervised** dimensionality reduction technique that:
- Finds **new axes** (principal components) that are **linear combinations** of your original features.
- Orders these axes so that:
  - **PC1** captures the **most variance** in the feature space,
  - **PC2** captures the next most variance, and so on,
  - Each PC is **orthogonal** (uncorrelated) with the previous ones.

### What PCA is *not*
PCA does **not** directly find features that “impact the target variable” because it does not use the target in its optimization.

However, you *can*:
- Compute PCs from the feature matrix **X** (after normalization),
- Then measure how PCs relate to the target **y** (e.g., correlation with `Price`, or a simple regression on PCs),
- Interpret which original features contribute most to PCs that are most related to **y**.

---

## 🧠 The Math (high level)
Given a centered feature matrix \(X\) (often standardized/normalized first):

1. Compute covariance matrix:
$$
\Sigma = \frac{1}{n-1}X^\top X
$$

2. Find eigenvectors (principal directions) and eigenvalues (variance captured):
$$
\Sigma v_i = \lambda_i v_i
$$

- $ v_i $ are **principal component directions** (loadings)
- $ \lambda_i $ are the **variance explained** by each component

---

## ✅ Why Normalize Before PCA?
PCA is sensitive to scale. Without normalization/standardization:
- A large-scale feature (e.g., `Price`) can dominate variance
- PCs will reflect units rather than structure

---

## 🎯 Student Challenge
Using the **housing dataset**:

1. Apply PCA to the normalized feature matrix \(X\) (exclude ID columns and the target).
2. Determine how many components are needed to explain **≥ 90%** of the variance.
3. Identify which original features contribute most to:
   - **PC1** and **PC2**, and
   - the **PC most correlated with the target** (`Price`).
4. Write a short interpretation:
   - “What does PC1 represent in housing terms?”
   - “Do the PCs that explain the most variance also relate most strongly to `Price`?”



### 🔗 How to Integrate This With Your Step 1 Normalization

- If you created normalized columns (e.g., `Area_sqft_norm`), use those in `candidate_features`.
- If you normalized in-place (overwriting original columns), you can use the original names.
- PCA should **not** include:
  - `House_ID` (identifier)
  - non-numeric categorical columns (unless encoded appropriately)
- Decide intentionally whether to include `Year_Built`:
  - It’s numeric, but it may behave differently than size/price-related features.

---

### ✅ Deliverable for the Challenge
Add a Markdown cell answering:

1. How many PCs explain at least **90%** variance?
2. Which features contribute most to **PC1** and **PC2**?
3. Which PC is most correlated with `Price`?
4. In plain language: what do you think PC1 represents?


In [30]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ---- Step A: Choose target and feature columns ----
TARGET = 'Price'
FEATURES = ['Area_sqft', 'Num_Bedrooms', 'Num_Bathrooms', 'Year_Built', 'Lot_Size']
X = df_norm[FEATURES]
y = df[TARGET]

# ---- Step B: Ensure scaling appropriate for PCA ----
print("Ranges after min-max:", (X.max() - X.min()).round(3).to_dict())
print("Means (PCA centers these):", X.mean().round(3).to_dict())

Ranges after min-max: {'Area_sqft': 1.0, 'Num_Bedrooms': 1.0, 'Num_Bathrooms': 1.0, 'Year_Built': 1.0, 'Lot_Size': 1.0}
Means (PCA centers these): {'Area_sqft': 0.415, 'Num_Bedrooms': 0.496, 'Num_Bathrooms': 0.483, 'Year_Built': 0.496, 'Lot_Size': 0.416}


#### Step C — Covariance matrix, then fit PCA

PCA starts from the covariance matrix: it shows how each pair of features moves together.

- **Raw covariance:** in mixed units, so Price's big numbers hide everything.
- **Min-max covariance:** all features on the same scale.
- **Correlation:** always between −1 and 1, easiest to read.

Use the dropdown to switch views. `viz.covariance_heatmap(insights, features=[...], min_abs=...)` filters the chart to chosen features and hides weak pairs. The explorer below has all the filters as controls.

In [31]:
insights = FeatureInsights(df[NUMERIC], df_norm)
fig = viz.covariance_heatmap(insights)
fig.show()

In [32]:
fig = viz.covariance_heatmap(insights, features=['Price', 'Area_sqft', 'Num_Bedrooms', 'Year_Built'], min_abs=0.03, start='correlation')
fig.show()

In [33]:
viz.covariance_explorer(insights)

In [34]:
insights.strongest_pairs()

,Feature A,Feature B,r
0,Num_Bedrooms,Year_Built,0.036364
1,Area_sqft,Num_Bedrooms,0.033654
2,Area_sqft,Year_Built,-0.033065
3,Price,Year_Built,-0.032315
4,Num_Bedrooms,Lot_Size,-0.029712


In [35]:
# ---- Step C: Fit PCA ----
pca = PCAAnalyzer(X, target=y).fit()
print("Covariance matrix Σ of the normalized features:")
print(pca.covariance.round(4))
pca.agreement()

Covariance matrix Σ of the normalized features:
               Area_sqft  Num_Bedrooms  Num_Bathrooms  Year_Built  Lot_Size
Area_sqft         0.0223        0.0018         0.0001     -0.0015   -0.0007
Num_Bedrooms      0.0018        0.1241         0.0025      0.0038   -0.0017
Num_Bathrooms     0.0001        0.0025         0.1705     -0.0015    0.0007
Year_Built       -0.0015        0.0038        -0.0015      0.0864   -0.0001
Lot_Size         -0.0007       -0.0017         0.0007     -0.0001    0.0276


,Manual eigen ratio,sklearn ratio,Abs. difference
PC1,0.3961,0.3961,1.665335e-16
PC2,0.2888,0.2888,5.551115e-17
PC3,0.1996,0.1996,8.326673e-17
PC4,0.0642,0.0642,6.938894e-17
PC5,0.0514,0.0514,6.938894e-17


**What the covariance tells us:** without scaling, Price's huge numbers hide everything else. After scaling, every value between two different features is close to zero. The strongest link in the whole dataset is only **r = 0.036**. In simple words, the features barely move together, so PCA won't be able to combine them much. Our manual PCA matches sklearn exactly.

#### Step D — Variance explained

In [36]:
# ---- Step D: Variance explained ----
k90 = pca.components_for(0.90)
print(f"Components needed for ≥ 90% variance: {k90}")
fig = viz.scree(pca)
fig.show()
pca.variance_table()

Components needed for ≥ 90% variance: 4


,Eigenvalue,Explained %,Cumulative %
PC1,0.17070,39.61,39.61
PC2,0.12445,28.88,68.49
PC3,0.08600,19.96,88.44
PC4,0.02766,6.42,94.86
PC5,0.02215,5.14,100.00


In [37]:
comparison = PCAAnalyzer.compare_scalings(df[FEATURES], X)
fig = viz.scaling_comparison(comparison)
fig.show()
(comparison * 100).round(1)

,Raw units,"Min-max [0, 1]",Standardized (z-score)
PC1,94.1,39.6,21.2
PC2,100.0,68.5,41.9
PC3,100.0,88.4,62.1
PC4,100.0,94.9,81.4
PC5,100.0,100.0,100.0


#### Step E — Loadings and feature importance

Loadings show how much each original feature contributes to each PC. Feature importance adds this up across all PCs, weighted by how much variance each PC explains.

In [38]:
# ---- Step E: Loadings (feature contributions to PCs) ----
loadings = pca.loadings()
fig = viz.loadings_heatmap(loadings)
fig.show()
for pc in ['PC1', 'PC2']:
    print(pc, '→', pca.top_contributors(pc))

PC1 → [('Num_Bathrooms', 0.999), ('Num_Bedrooms', 0.052)]
PC2 → [('Num_Bedrooms', 0.993), ('Year_Built', 0.1)]


In [39]:
fig = viz.biplot(pca, y)
fig.show()

In [40]:
variance_share = pca.variance_importance()
impact = insights.price_impact()
fig = viz.importance_lenses(variance_share, impact)
fig.show()
(variance_share * 100).round(1).rename('Variance share (%)').to_frame()

,Variance share (%)
Num_Bathrooms,39.6
Num_Bedrooms,28.8
Year_Built,20.0
Lot_Size,6.4
Area_sqft,5.2


#### Step F — Relate the PCs and features to Price

First we check how each PC correlates with Price. Then we measure each feature's impact: after min-max, a regression coefficient means *"how much the predicted price changes going from the feature's lowest to highest value."* The bars show 95% ranges from 500 bootstrap resamples.

In [41]:
# ---- Step F: Relate PCs to the target (simple correlation) ----
pc_corr = pca.target_correlation()
best_pc, best_r = pca.most_target_related()
print(f"Most Price-related component: {best_pc} (r = {best_r:+.4f}), driven by {pca.top_contributors(best_pc, 1)}")
fig = viz.target_correlation(pc_corr)
fig.show()

Most Price-related component: PC3 (r = -0.0316), driven by [('Year_Built', 0.995)]


In [42]:
fig = viz.price_impact(impact, insights.r2)
fig.show()
print(f"R² of Price on all five normalized features: {insights.r2:.4f}")
impact.style.format({'Impact ($, min to max)': '${:,.0f}', 'CI low': '${:,.0f}', 'CI high': '${:,.0f}', 'Correlation r': '{:+.4f}'})

R² of Price on all five normalized features: 0.0018


,"Impact ($, min to max)",CI low,CI high,Correlation r,CI crosses 0
Area_sqft,"$-18,005","$-63,118","$22,960",-0.0177,True
Year_Built,"$-16,550","$-37,783","$5,607",-0.0323,True
Lot_Size,"$13,267","$-23,232","$45,785",+0.0155,True
Num_Bathrooms,"$-4,160","$-19,376","$13,268",-0.0112,True
Num_Bedrooms,"$-1,433","$-19,663","$17,708",-0.0059,True


**What the impact chart tells us:** Every range crosses $0 and R² is only **0.0018**. In simple words: none of these features can reliably predict Price in this dataset. A real price model would need other information, like location.

### ✅ Challenge Deliverable

We kept `Year_Built` because it is a real house feature. It does not affect the other features.

1. **How many PCs explain at least 90% variance?** **4 PCs.** Three only reach 88.4%; four reach 94.9%.
2. **Which features drive PC1 and PC2?** PC1 is almost all **bathrooms**. PC2 is almost all **bedrooms**.
3. **Which PC is most related to Price?** **PC3** (mostly `Year_Built`), but the link is tiny (r = −0.032).
4. **What does PC1 represent?** Simply **"number of bathrooms."** The features don't move together, so each PC ends up being mostly one feature.

**Do the biggest PCs relate most to Price?** No. PC1 and PC2 hold the most variance but have almost no link to Price.

### 🔎 Talking Points #2 — What PCA, covariance and feature impact taught us

- **Scale before PCA:** On raw data, PC1 seems to explain 94% of the variance, but only because Lot_Size has big numbers. After scaling, we need 4 PCs for 90%.
- **PCA needs related features:** Our features are almost unrelated, so PCA can't combine them. Each PC is basically just one feature.
- **More variance does not mean more important:** The PCs with the most variance have almost no link to Price. PCA ignores the target, so we have to check the link to Price separately.

# 📤 How to Submit the Assignment

Follow these instructions carefully to ensure your team submits a complete and well-organized assignment.

## Step 1: Get the Starter Repository

1. Clone the GitHub repository provided by the instructor.
2. Keep the **original repository name**. Do **not** rename the repository.

## Step 2: Complete the Workshop

3. Create **one shared Jupyter Notebook** for the entire team to work on.
4. At the top of **`MinMax_Normalization_Workshop.ipynb`**, clearly include:
   - Team Number
   - Full Name of each team member
5. Complete **all coding tasks, analyses, and written answers** in the same notebook.
6. Make sure all notebook cells execute successfully before submission.

## Step 3: Update the Project Documentation

7. Replace the existing **`README.md`** file with a new version that:
   - Introduces your team
   - Explains the purpose of the workshop
   - Describes the work you completed
   - Summarizes your findings and conclusions
   - Can be used as the visual aid for your team's **5-minute presentation**

## Step 4: Review Project Dependencies

8. Review the project's Python dependencies and decide whether the **`requirements.txt`** file needs to be updated.
   - If you installed additional packages, regenerate or update the file.
   - If no new dependencies were added, no changes may be necessary.

## Step 5: Publish Your Work

9. Push the completed repository to your own GitHub account.
10. Verify that all files have been uploaded correctly and that the repository is publicly accessible if required by the instructor.

---

## ✅ Submission Checklist

Before submitting, confirm that the following items are complete:

- [ ] Repository name has not been changed
- [ ] `MinMax_Normalization_Workshop.ipynb` contains all required work
- [ ] Team number and team member names appear at the top of the notebook
- [ ] All questions have been answered
- [ ] All code runs successfully
- [ ] `README.md` has been replaced with a project summary
- [ ] README can support a 5-minute presentation
- [ ] `requirements.txt` has been reviewed and updated if necessary
- [ ] Repository has been pushed to your GitHub account
- [ ] Final work has been reviewed by all team members

> **Important:** Submit a repository that is clean, complete, and easy to understand. The notebook and README should clearly demonstrate your team's work, findings, and understanding of Min-Max Normalization.